In [7]:
# Core Features for ABC-XYZ ML Model
features = {
    # ABC Features (Value-based)
    'revenue_contribution': 'monthly_revenue / total_revenue',
    'profit_contribution': 'monthly_profit / total_profit',
    'margin_percentage': '(price - cost) / price',
    'inventory_turnover': 'COGS / average_inventory',
    'sku_velocity': 'units_sold / days_active',
    
    # XYZ Features (Variability-based)
    'coefficient_variation': 'std_demand / mean_demand',
    'adi': 'average_inter_demand_interval',  # days between orders
    'cv_squared': 'coefficient_of_variation^2',
    'demand_trend': 'slope_of_12_month_regression',
    'seasonal_strength': 'variance_explained_by_seasonality',
    'forecast_error_mape': 'mean_absolute_percentage_error',
    
    # Hybrid Features
    'revenue_stability': 'revenue_contribution * (1 - cv)',
    'strategic_importance': 'composite_score_lead_time_criticality'
}

In [2]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np

class ABCSmartClassifier:
    def __init__(self, n_clusters=3):
        self.scaler = StandardScaler()
        self.kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        self.feature_weights = {
            'revenue_contribution': 0.4,
            'profit_contribution': 0.3,
            'inventory_turnover': 0.2,
            'strategic_score': 0.1
        }
    
    def fit(self, df):
        # Weighted feature matrix
        X = self._create_weighted_features(df)
        X_scaled = self.scaler.fit_transform(X)
        self.kmeans.fit(X_scaled)
        
        # Map clusters to A, B, C based on centroids
        centroids = self.kmeans.cluster_centers_
        # Sort by revenue weight and assign labels
        sorted_idx = np.argsort([c[0] for c in centroids])[::-1]
        self.cluster_map = {sorted_idx[0]: 'A', 
                           sorted_idx[1]: 'B', 
                           sorted_idx[2]: 'C'}
        return self
    
    def predict(self, df):
        X = self._create_weighted_features(df)
        X_scaled = self.scaler.transform(X)
        clusters = self.kmeans.predict(X_scaled)
        return [self.cluster_map[c] for c in clusters]
    
    def _create_weighted_features(self, df):
        # use df.get so that missing columns default to zero; ensure each
        # returned object is a 1‑D array of length len(df)
        arrs = []
        for col, weight in self.feature_weights.items():
            val = df.get(col, 0)
            if np.isscalar(val):
                # create full-length series of the scalar
                arr = np.full(len(df), val)
            else:
                arr = np.asarray(val)
            arrs.append(arr * weight)
        features = np.column_stack(arrs)
        return features

In [3]:
class DemandPatternClassifier:
    """
    ML-enhanced SBC classification with additional pattern detection
    """
    def __init__(self):
        self.scaler = StandardScaler()
        self.classifier = RandomForestClassifier(n_estimators=100)
        
    def calculate_sbc_metrics(self, demand_series):
        """
        Calculate ADI and CV² for demand series
        """
        # ADI: Average Inter-Demand Interval
        non_zero_demand = demand_series[demand_series > 0]
        if len(non_zero_demand) == 0:
            return {'adi': np.inf, 'cv2': 0}
        
        intervals = np.diff(np.where(demand_series > 0)[0])
        adi = np.mean(intervals) if len(intervals) > 0 else 0
        
        # CV²: Squared Coefficient of Variation
        cv = np.std(non_zero_demand) / np.mean(non_zero_demand)
        cv2 = cv ** 2
        
        return {'adi': adi, 'cv2': cv2}
    
    def extract_advanced_features(self, demand_series):
        """
        Extract ML features beyond basic SBC
        """
        features = {}
        
        # Basic SBC
        sbc = self.calculate_sbc_metrics(demand_series)
        features.update(sbc)
        
        # Advanced time-series features
        features['trend_strength'] = self._calculate_trend(demand_series)
        features['seasonal_strength'] = self._calculate_seasonality(demand_series)
        features['zero_demand_ratio'] = np.sum(demand_series == 0) / len(demand_series)
        features['demand_entropy'] = self._calculate_entropy(demand_series)
        features['burstiness'] = self._calculate_burstiness(demand_series)
        features['autocorr_lag1'] = pd.Series(demand_series).autocorr(lag=1)
        
        return features

    # helper stubs for time‑series characteristic calculations
    def _calculate_trend(self, series):
        # placeholder: could compute slope of regression
        return 0

    def _calculate_seasonality(self, series):
        # placeholder: compute strength of seasonal component
        return 0

    def _calculate_entropy(self, series):
        # placeholder: entropy of distribution
        return 0

    def _calculate_burstiness(self, series):
        # placeholder: measure of sporadic spikes
        return 0
    
    def _classify_sbc_rule(self, adi, cv2):
        """Traditional SBC classification"""
        if adi < 1.32 and cv2 < 0.49:
            return 'Smooth'
        elif adi >= 1.32 and cv2 < 0.49:
            return 'Intermittent'
        elif adi < 1.32 and cv2 >= 0.49:
            return 'Erratic'
        else:
            return 'Lumpy'
    
    def fit_ml_enhanced(self, training_data):
        """
        Train ML model to improve upon SBC rules
        """
        # Prepare training data
        X = []
        y = []
        
        for series in training_data:
            features = self.extract_advanced_features(series)
            X.append(list(features.values()))
            # Use SBC rule as baseline label, can be enhanced with expert labels
            y.append(self._classify_sbc_rule(features['adi'], features['cv2']))
        
        X = np.array(X)
        y = np.array(y)
        
        X_scaled = self.scaler.fit_transform(X)
        self.classifier.fit(X_scaled, y)
        
    def predict(self, demand_series):
        features = self.extract_advanced_features(demand_series)
        X = np.array(list(features.values())).reshape(1, -1)
        X_scaled = self.scaler.transform(X)
        
        # ML prediction with probability
        pred_proba = self.classifier.predict_proba(X_scaled)[0]
        pred_class = self.classifier.predict(X_scaled)[0]
        
        # Confidence score
        confidence = max(pred_proba)
        
        return {
            'pattern': pred_class,
            'confidence': confidence,
            'features': features,
            'probabilities': dict(zip(self.classifier.classes_, pred_proba))
        }

In [4]:
def assign_forecasting_method(abc_class, xyz_class, ml_confidence):
    """
    Assign optimal forecasting method based on ABC-XYZ matrix
    """
    method_matrix = {
        ('A', 'X'): {'method': 'ARIMA/ETS', 'review_period': 'Daily', 'safety_stock_days': 7},
        ('A', 'Y'): {'method': 'XGBoost/LightGBM', 'review_period': 'Daily', 'safety_stock_days': 14},
        ('A', 'Z'): {'method': 'Ensemble (ML + Safety Stock)', 'review_period': 'Daily', 'safety_stock_days': 30},
        ('B', 'X'): {'method': 'Exponential Smoothing', 'review_period': 'Weekly', 'safety_stock_days': 14},
        ('B', 'Y'): {'method': 'Prophet/Seasonal ML', 'review_period': 'Weekly', 'safety_stock_days': 21},
        ('B', 'Z'): {'method': 'Croston/SBA', 'review_period': 'Bi-weekly', 'safety_stock_days': 45},
        ('C', 'X'): {'method': 'Moving Average', 'review_period': 'Monthly', 'safety_stock_days': 30},
        ('C', 'Y'): {'method': 'Simple Exponential Smoothing', 'review_period': 'Monthly', 'safety_stock_days': 45},
        ('C', 'Z'): {'method': 'Min-Max Inventory', 'review_period': 'Quarterly', 'safety_stock_days': 90}
    }
    
    base_method = method_matrix.get((abc_class, xyz_class))
    
    # Adjust based on ML confidence
    if ml_confidence < 0.6:
        base_method['method'] += ' + Conservative Safety Stock'
        base_method['safety_stock_days'] *= 1.5
    
    return base_method

In [5]:
import pandas as pd
from datetime import datetime

# --- load the PO data ------------------------------------------------------
path = r'Jan 1st week Spare Parts PO.xlsx'
po = pd.read_excel(path)

# inspect columns so we know what names to use
print("columns:", list(po.columns))

# adapt column names below to whatever your sheet uses
# --------------------------------------------------------------------------
# determine candidates and raise an error if none of the expected names are
candidates = {
    'revenue': ['revenue', 'Revenue', 'Net Value (Item)', 'Net Value (Header)', 'Net Price'],
    'profit':  ['profit', 'Profit'],
    'price':   ['price', 'Price', 'Net Price'],
    'cost':    ['cost', 'Cost'],
    'cogs':    ['COGS', 'Cogs'],
    'avg_inv': ['avg_inventory', 'AvgInventory'],
    'units':   ['units_sold', 'Order Quantity (Item)', 'Target Quantity'],
    'days':    ['days_active']
}

def find_col(options):
    for o in options:
        if o in po.columns:
            return o
    return None

rev_col = find_col(candidates['revenue'])
profit_col = find_col(candidates['profit'])
price_col = find_col(candidates['price'])
cost_col = find_col(candidates['cost'])
cogs_col = find_col(candidates['cogs'])
avg_inv_col = find_col(candidates['avg_inv'])
units_col = find_col(candidates['units'])
days_col = find_col(candidates['days'])

# if we didn't find profit we can proceed with revenue only
if rev_col is None:
    raise ValueError(
        "Could not identify any revenue-like column in PO file. "
        "Please inspect the printed column list and update the mapping."
    )

# example aggregations, adjust to your real columns:
total_revenue = po[rev_col].sum()
po['revenue_contribution']  = po[rev_col] / total_revenue

if profit_col is not None:
    total_profit  = po[profit_col].sum()
    po['profit_contribution']   = po[profit_col] / total_profit
else:
    # profit not available; fill with zeros so ABC model can still run
    po['profit_contribution'] = 0

if price_col and cost_col in po.columns:
    po['margin_percentage']     = (po[price_col] - po[cost_col]) / po[price_col]
if cogs_col and avg_inv_col:
    po['inventory_turnover']    = po[cogs_col] / po[avg_inv_col]
if units_col and days_col:
    po['sku_velocity']          = po[units_col] / po[days_col]
# …and so on for any other ABC features

# --- build demand history for XYZ pattern --------------------------------
# choose columns for SKU, order date and quantity
sku_col = find_col(['sku', 'Material', 'Material Description'])
date_col = find_col(['order_date', 'Document Date', 'Delivery Date', 'Created On'])
qty_col = find_col(['quantity', 'Order Quantity (Item)', 'Target Quantity', 'Confirmed Quantity (Item)'])

if sku_col is None or date_col is None or qty_col is None:
    print("could not locate sku/date/quantity columns; demand matrix will be empty")
    demand = pd.DataFrame()
else:
    demand = (po
              .groupby([sku_col, date_col])[qty_col]
              .sum()
              .unstack(fill_value=0))       # rows = sku, cols = dates

columns: ['Customer Reference', 'Document Date', 'Sales Document Type', 'Sales Document', 'Sales Document Item', 'Sold-to Party', 'Material', 'Order Quantity (Item)', 'Sales Unit', 'Net Value (Item)', 'Document Currency', 'Order Reason', 'Customer Reference (Header)', 'Created By', 'Created On', 'Time', 'Billing Block', 'Sales Document Description', 'Exchange Rate Type', 'Delivery Block', 'Net Value (Header)', 'Division', 'SD Document Category', 'Sales Office', 'Sales Group', 'Sales Organization', 'Distribution Channel', 'Overall Status', 'Overall Delivery Status (All Items)', 'Sold-To Party Name', 'Reason for Rejection', 'Material Description', 'Batch', 'Confirmed Quantity (Item)', 'Pricing Unit', 'Pricing Unit (per)', 'Storage Location', 'Base Unit of Measure', 'Net Price', 'Returns', 'Shipping Point/Receiving Pt', 'Plant', 'Target Quantity', 'Overall Status Item', 'Overall Delivery Status (Item)', 'Exchange Rate', 'Pricing Date', 'Personnel Partner Function', 'Personnel Number', 'Pa

In [10]:
# import classifier classes from earlier cells in this notebook
# (they are defined above; no extra import required)

# 2‑a. ABC classification
abc_model = ABCSmartClassifier()        # default 3 clusters = A/B/C
abc_model.fit(po)                      # pass the dataframe with ABC features
po['abc_class'] = abc_model.predict(po)

# 2‑b. XYZ classification (demand‑pattern)
xyz_model = DemandPatternClassifier()

# if we have demand history, fit the XYZ classifier using those series
if not demand.empty:
    sku_col = find_col(['sku', 'Material', 'Material Description'])
    # demand index contains SKUs
    demand_map = {sku: demand.loc[sku].values for sku in demand.index}

    # training data: just use the historical series themselves
    xyz_model.fit_ml_enhanced(list(demand_map.values()))

    def get_xyz(sku):
        series = demand_map.get(sku, np.zeros(demand.shape[1]))
        return xyz_model.predict(series)['pattern']

    po['xyz_class'] = po[sku_col].map(get_xyz)
else:
    po['xyz_class'] = None

# 2‑c. combine both
po['abc_xyz'] = po['abc_class'].astype(str) + '-' + po['xyz_class'].astype(str)

d:\anaconda\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=7.
  warnings.warn(
d:\anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
d:\anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
d:\anaconda\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\an